<a href="https://colab.research.google.com/github/mmera886/Entrega-Seguimiento-2/blob/main/EA1_Ingesti%C3%B3n_de_Datos_desde_un_API_(2)_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**<span style="color:#809bd8">Curso:</span>** **Introducción a Ingeniería de Sistemas**

**<span style="color:#809bd8">Tema:</span>** **Evidencia de Aprendizaje EA1**

**<span style="color:#809bd8">Estudiante:</span>** **Miguel Angel Mera Carreño**

**<span style="color:#809bd8">Profesor:</span>** **Walter Hugo Arboleda Mazo**

**<span style="color:#809bd8">Universidad:</span>** **UNAC**

**<span style="color:#809bd8">Fecha:</span>** **25/02/2026**

In [ ]:
#pip install requests pandas openpyxl
#Este mensaje indica que las librerías requests, pandas y openpyxl ya están instaladas en el sistema, por lo que no es necesario volver a instalarlas.
#Tambien muestra el lugar en donde estan instaladas.


In [ ]:
import requests # importacion de la libreria Requests para enviar peticion a la API https://jsonplaceholder.typicode.com/users
import sqlite3 # importacion de la interface para conexion y creacion de bases de datos SQLite
import pandas as pd # importacion de Pandas, libreria para el analisis de datos en Python

# Definicion de la URL de la API para envio de peticion y recepcion de datos usando protocolo HTTP
# desde la API https://jsonplaceholder.typicode.com/
API_URL = "https://jsonplaceholder.typicode.com/users"


# Creacion de la funcion extraer_datos validandose codigo 200 (exito en la conexion) o codigo 404 (codigo de error en la conexion)
def extraer_datos(url):
    response = requests.get(url) #se realiza la peticion HTTP
    if response.status_code == 200:
        return response.json() #devuelve los datos en formato JSON (formato de texto ligero, estándar y fácil de leer)
    else:
        raise Exception(f"Error al conectar con la API: {response.status_code}")


# Creacion de la base de datos en SQLite "datos_api.db"
# se realiza la conexion a dicha base de datos
# y se crea el "cursor" para "execute" el CREATE TABLE y el INSERT OR REPLACE INTO usuarios
def guardar_en_db(datos):
    conn = sqlite3.connect('datos_api.db')
    cursor = conn.cursor()

    # Creación de la tabla "usuarios" dentro de la base de datos "datos_api.db"
    # esta contiene los campos id, nombre, usuario y email
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS usuarios (
            id INTEGER PRIMARY KEY,
            nombre TEXT,
            usuario TEXT,
            email TEXT
        )
    ''')

    # Inserción de datos en la tabla "usuarios"
    for item in datos:
      #Funcion de insertar o reemplazar en la tabla de usuarios dependiendo su clasificacion
        cursor.execute('''
            INSERT OR REPLACE INTO usuarios (id, nombre, usuario, email)
            VALUES (?, ?, ?, ?)
        ''', (item['id'], item['name'], item['username'], item['email']))

    # Aceptación mediante "Commit" el ingreso de datos en la tabla "usuarios"
    # Commit es una confirmación de los cambios realizados en tus archivos en un momento específico)
    conn.commit()

    # Cerrado de la conexion a la base de datos "datos_api.db"
    conn.close()

# 3. GENERAR EVIDENCIAS (Pandas y Auditoría)
def generar_evidencias(datos_api):
    # --- Archivo Excel/CSV con Pandas ---
    conn = sqlite3.connect('datos_api.db')
    df_db = pd.read_sql_query("SELECT * FROM usuarios", conn)
    df_db.to_csv('muestra_usuarios.csv', index=False)
    conn.close()

    # la variable "registros_api" obtiene la cantidad de registros recibidos desde la
    # API con url "https://jsonplaceholder.typicode.com/users"
    # la variable "registros_db" obtiene la cantidad de registros almacenados en la tabla "usuarios"
    registros_api = len(datos_api)
    registros_db = len(df_db)

    # Creación del archivo "auditoría.txt" con el metodo "open" de Python
    with open('auditoria.txt', 'w', encoding='utf-8') as f:
        f.write("RESUMEN DE AUDITORÍA\n")
        f.write("====================\n")
        f.write(f"Registros extraídos de la API https://jsonplaceholder.typicode.com/: {registros_api}\n")
        f.write(f"Registros guardados en la DB de SQLite3: {registros_db}\n")

        if registros_api == registros_db:
            f.write("\nESTADO: Operacion Exitosa.\n")
        else:
            f.write("\nESTADO: Operacion Fallida.\n")

# EJECUCIÓN DEL PROCESO
try:
    print("Iniciando proceso de petición y recepción de datos desde la API https://jsonplaceholder.typicode.com/users...")
    datos = extraer_datos(API_URL)

    print("Creando la base de datos datos_api.db y guardando datos en la tabla usuarios...")
    guardar_en_db(datos)

    print("Generando archivos de evidencia .CSV y auditoría.txt...")
    generar_evidencias(datos)

    print("¡Proceso completado con éxito!")
except Exception as e:
    print(f"Ocurrió un error: {e}")


#==================Investigacion acerca de las librerias del codigo=============================================================================

#Requests Library de python:
#La Requests Library de Python es una librería que se utiliza para enviar solicitudes HTTP de forma sencilla desde un programa en Python.
#Permite comunicarse con APIs, páginas web o servidores, obteniendo o enviando datos a través de internet.

#Libreria Pandas:
#La librería Pandas de Python es una herramienta utilizada para analizar, organizar y manipular datos de forma eficiente.
#Sirve para Leer archivos CSV, Excel, JSON o bases de datos, organizar tablas, Filtrar información, Limpiar datos, entre otras...

#Openpyxl:
#Openpyxl es una librería de Python que permite leer, crear y modificar archivos de Excel (.xlsx) directamente desde un programa.
#Se usa cuando se necesita automatizar tareas con hojas de cálculo sin tener que abrir Excel manualmente.
# Permite automatizar Excel con Python, es compatible con archivos .xlsx y permite leer y modificar datos fácilmente

#Sqlite:
# SQLite es un sistema de gestión de bases de datos relacional (RDBMS) que permite almacenar y gestionar datos en forma de tablas.
# Es similar a otras bases de datos como MySQL o PostgreSQL, pero con la ventaja de que es muy ligero y no necesita un servidor.
# Crear bases de datos locales, guarda información estructurada en tablas, consulta datos con SQL y desarrolla aplicaciones pequeñas o prototipos.

#SQLite Viewer Web App:
#SQLite Viewer Web App es una aplicación web que permite abrir y explorar archivos de bases de datos
#Sirve principalmente para visualizar el contenido de una base de datos SQLite, ver sus tablas y revisar los datos que contiene.}
#Sus funciones son: Abrir archivos .db o .sqlite, Ver las tablas de la base de datos, Explorar los registros almacenados, etc..

#SQLite 3:
#SQLite3 es la versión más utilizada del sistema de base de datos SQLite y también el módulo de Python
#SQLite3 permite crear, modificar y consultar bases de datos directamente desde Python usando comandos SQL.
#Es una libreria incluida por defecto que permite: Crear bases de datos, crear tablas, insertar datos, etc.





Iniciando proceso de petición y recepción de datos desde la API https://jsonplaceholder.typicode.com/users...
Creando la base de datos datos_api.db y guardando datos en la tabla usuarios...
Generando archivos de evidencia .CSV y auditoría.txt...
¡Proceso completado con éxito!



**Referencias**

https://realpython.com/python-requests/#make-a-get-request

https://pandas.pydata.org/

https://pypi.org/project/openpyxl/

https://docs.python.org/3/library/sqlite3.html

https://sqliteviewer.app/

https://sibabalwesinyaniso.medium.com/what-is-a-cursor-understanding-how-sqlite-handles-queries-in-python-8f8c88546820
https://www.pythonlore.com/understanding-sqlite3-cursor-object-for-database-operations/
https://miro.medium.com/v2/resize:fit:720/format:webp/0*rePKZ6jMZbZ_6S_3.gif
